# data 

## load

In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler


from src.utils import *
import src.prompt as prompt
from src.data_loader import load_spatial_data_anndata

In [ ]:
representetive_gene_list = (repository_root() / "examples/merfish/representative_genes.txt").read_text().splitlines()


In [ ]:
config = load_config("configs/config_zeroshot_merfish.yaml")
config.data_name = "MERFISH_25"
# config.replicate = "_rep14o"
config.refresh_paths()


In [ ]:
# --- Load data ---
data_path = str(dataset_root("merfish"))

# rename cell types
rename_celltype = True
celltype_rename = {
    'Astrocyte' : 'Astrocyte',
 'Endothelial 1': 'Endothelial',
 'OD Mature 2': 'Mature oligodendrocytes',
 'Inhibitory': 'Inhibitory',
 'OD Immature 1': 'Immature oligodendrocytes',
 'Excitatory': 'Excitatory',
 'Endothelial 3': 'Endothelial',
 'Microglia': 'Microglia',
 'OD Mature 1': 'Mature oligodendrocytes',
 'Pericytes': 'Pericytes',
 'OD Mature 4': 'Mature oligodendrocytes',
 'Endothelial 2': 'Endothelial',
 'OD Mature 3': 'Mature oligodendrocytes',
 'OD Immature 2': 'Immature oligodendrocytes',
 'Ependymal': 'Ependymal'
}
config.celltype_rename = celltype_rename


# Load data using the new function
adata = load_spatial_data_anndata(
    data_path=data_path,
    adata_file=f"{config.data_name}.h5ad",
    config=config,
    rename_celltype=rename_celltype 
)

In [ ]:
# Prepare neighbor data using the new function
neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
    adata, config, representetive_gene_list
)

# prompt

full_domain_name = {"BST": "bed nuclei of the strata terminalis", 
 "V3": "third ventricle", 
 "PV": "periventricular hypothalamic nucleus", 
 "PVT": "paraventricular nucleus of the thalamus", 
 "MPN": "medial preoptic nucleus", 
 "fx": "columns of the fornix",
 "PVH": "paraventricular hypothalamic nucleus",
 "MPA": "medial preoptic area"}
adata.obs[name_truth] = adata.obs[name_truth].map(full_domain_name)

In [ ]:
unique_layers = adata.obs[config.name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

# # the full cell type names are already in the data
# unique_celltypes = adata.obs[config.celltype_name].unique()
# cell_names_mapping = {celltype: celltype for _, celltype in enumerate(unique_celltypes)}

config.domain_mapping = domain_mapping
# config.cell_names_mapping = cell_names_mapping



In [ ]:

# IMPORTANT: check input and prompt_func
if config.Graph_type == "countPlusGenes":
    input_df = neighbor_normalized_df
    df_extra = neighbor_normalized_df_genes
    prompt_func = prompt.zeroshot_celltype_geneorder
elif config.Graph_type == "count":
    input_df = neighbor_normalized_df
    df_extra = None
    prompt_func = prompt.zeroshot_celltype
elif config.Graph_type == "GeneOnly":
    input_df = neighbor_normalized_df_genes
    df_extra = None
    prompt_func = prompt.zeroshot_geneorder
else:
    raise ValueError(f"Graph_type {config.Graph_type} not supported")

In [ ]:
x = [i for i in range(len(adata)) if adata.obs[config.name_truth].iloc[i] == "fx"][50:51]

print(prompt_func(input_df, x, config))

# gpt

## generate json

In [ ]:
print(f"{config.gpt_model}")

In [ ]:
# choose correct data and prompt
generate_json_end2end(input_df, 
                      config, 
                      prompt_func, 
                      batch_size = 2000,
                      max_completion_tokens = 512,  # key to control the cost, expecially for o3-mini
                      n_rows = 1,
                      df_extra = df_extra)

## submit

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_zeroshot_merfish.yaml {config.data_name} {config.replicate} parallel > outs/zeroshot_{config.data_name}{config.replicate}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# retrive the results of the parallel runs
cmd = f"nohup python -u -m src.retrive_batch_results_parallel configs/config_zeroshot_merfish.yaml {config.data_name} {config.replicate} 20250821_09 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 3
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']

                # extract outputs - handle both JSON format and text format
                extract_dict = extract_json_microenvironment(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['zeroshot_gpt4o_mini']

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.zeroshot_gpt4o_mini.value_counts()

In [ ]:
# # if the number of the cell type is less than 4, set it to unknown
# for nichtype in gpt_results_df.zeroshot_gpt4o_mini.value_counts()[gpt_results_df.zeroshot_gpt4o_mini.value_counts()<4].index:
#     gpt_results_df.loc[gpt_results_df.zeroshot_gpt4o_mini == nichtype, "zeroshot_gpt4o_mini"] = "unknown"


# Gemini


In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)


In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                neighbor_normalized_df, config, 
                                                prompt.zeroshot_celltype_geneorder, n_rows=1, 
                                                df_extra = neighbor_normalized_df_genes,
                                                column_name="zeroshot_gemini")

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
gemini_results_df.index = range(len(gemini_results_df))
gemini_results_df.drop(index=[971], inplace=True)

In [ ]:
gemini_results_df = pd.read_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)
gemini_results_df.index = gemini_results_df.index.astype(str)

In [ ]:
gemini_results_df.index = adata.obs.index

In [ ]:
gemini_results_df.zeroshot_gemini.value_counts()



In [ ]:
gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

# plot and save

In [ ]:
#adata.obs = adata.obs.join(gemini_results_df)
adata.obs = adata.obs.join(gpt_results_df)
#adata.obs['zeroshot_gemini'] = adata.obs['zeroshot_gemini'].fillna("unknown")
adata.obs['zeroshot_gpt4o_mini'] = adata.obs['zeroshot_gpt4o_mini'].fillna("unknown")
#sc.pl.scatter(adata, x="x", y="y", color="zeroshot_gemini", title =  f"zeroshot_gemini")
sc.pl.scatter(adata, x="x", y="y", color="zeroshot_gpt4o_mini", title =  f"zeroshot_gpt4o_mini")
#print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_gemini']))
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini']))


In [ ]:
normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini'])

In [ ]:
gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

# refine the niche

In [ ]:

#adata.obs['zeroshot_gemini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_gemini'])
adata.obs['zeroshot_gpt4o_mini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_gpt4o_mini'])



In [ ]:
adata.obs.to_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")